# 📊 Analyse des performances des modèles (Cross-Validation)
Ce notebook charge les résultats exportés par notre pipeline (fichiers `.csv`) et génère les visualisations comparatives.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os

# ============================================================
# CONFIGURATION DES FICHIERS
# ============================================================
# /!\ Vérifie que ces chemins correspondent exactement à ton architecture
METRICS_CSV = "Cross_val/results/cv_metrics_summary.csv"
PREDICTIONS_CSV = "CrossVal/results/cv_predictions.csv"
OUTPUT_DIR = "plots_output"

# Couleurs et décalages verticaux pour le graphique F1
STYLE_CONFIG = {
    "DINO_SVM":    {"color": "#4C72B0", "offset": -0.3},
    "DINO_LogReg": {"color": "#DD8452", "offset": -0.15},
    "DINO_MLP":    {"color": "#55A868", "offset": 0.0},
    "LightGBM":    {"color": "#E8DA7D", "offset": 0.15}, 
    "PointNet":    {"color": "#C44E52", "offset": 0.3}
}

# Création du dossier d'export
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"📁 Dossier '{OUTPUT_DIR}' créé avec succès.")
else:
    print(f"📁 Dossier '{OUTPUT_DIR}' prêt.")

## 1. Comparaison Globale des F1-Scores
Ce graphique affiche les performances de tous les modèles superposés, avec leurs intervalles de confiance sur 25 folds.

In [ ]:
def plot_f1_comparison():
    if not os.path.exists(METRICS_CSV):
        print(f"❌ Fichier introuvable : {METRICS_CSV}")
        return

    df = pd.read_csv(METRICS_CSV)
    
    # On isole les lignes "per_species"
    df_species = df[df['scope'] == 'per_species'].copy()
    
    # Ordre de tri basé sur la moyenne globale de tous les modèles
    species_order = df_species.groupby('name')['mean'].mean().sort_values().index.tolist()
    
    fig, ax = plt.subplots(figsize=(14, 16))
    plt.style.use('seaborn-v0_8-whitegrid')
    
    y_pos = np.arange(len(species_order))
    
    for method, style in STYLE_CONFIG.items():
        df_method = df_species[df_species['method'] == method].set_index('name')
        
        if df_method.empty:
            continue
            
        df_method = df_method.reindex(species_order)
        
        y_shifted = y_pos + style['offset']
        means = df_method['mean'].values
        ci_low = means - df_method['ci_low_95'].values
        ci_high = df_method['ci_high_95'].values - means
        
        ax.errorbar(means, y_shifted, xerr=[ci_low, ci_high], fmt='none', 
                    ecolor=style['color'], elinewidth=1.2, capsize=3, zorder=1, alpha=0.6)
        
        ax.scatter(means, y_shifted, color=style['color'], 
                   s=60, edgecolors='white', linewidth=0.5, zorder=2, label=method)
        
        global_f1 = df[(df['method'] == method) & (df['name'] == 'Macro F1')]['mean']
        if not global_f1.empty:
            ax.axvline(global_f1.values[0], color=style['color'], linestyle='--', linewidth=1.2, alpha=0.5, zorder=0)

    # Esthétique
    ax.set_yticks(y_pos)
    ax.set_yticklabels(species_order, fontsize=11, fontstyle='italic')
    ax.set_xlim(-0.02, 1.05)
    ax.set_xlabel('F1-Score (avec Intervalle de Confiance 95%)', fontsize=12, fontweight='bold', labelpad=15)
    ax.legend(title="Modèles évalués", loc='lower right', fontsize=11, title_fontsize=12, framealpha=0.95, edgecolor='black')
    
    for y in y_pos:
        ax.axhline(y, color='gray', linestyle='-', alpha=0.15, zorder=0)
        
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', linestyle=':', color='#e0e0e0', alpha=0.7)
    ax.grid(axis='y', visible=False)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.suptitle('Comparaison des Modèles : F1-Score par Espèce d\'Arbre (25 Folds)', fontsize=16, fontweight='bold')
    
    # Sauvegarde ET affichage dans le notebook
    out_path = os.path.join(OUTPUT_DIR, "f1_scores_comparison.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✅ F1-Scores enregistrés sous : {out_path}")
    plt.show()

# Lancement de la fonction
plot_f1_comparison()

## 2. Matrices de Confusion
Génération d'une heatmap de confusion pour chaque modèle afin de visualiser les erreurs de prédictions inter-espèces.

In [ ]:
def plot_confusion_matrices():
    if not os.path.exists(PREDICTIONS_CSV):
        print(f"❌ Fichier introuvable : {PREDICTIONS_CSV}")
        return

    df = pd.read_csv(PREDICTIONS_CSV)
    
    # Identifier les colonnes de prédiction
    pred_columns = [col for col in df.columns if col.startswith('pred_')]
    all_classes = sorted(df['true_species'].unique().tolist())
    
    for pred_col in pred_columns:
        method_name = pred_col.replace('pred_', '')
        
        cm = confusion_matrix(df['true_species'], df[pred_col], labels=all_classes)
        
        # Pourcentage par ligne pour normaliser les couleurs
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm_norm = np.nan_to_num(cm_norm) 
        
        plt.figure(figsize=(16, 14))
        sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues",
                    xticklabels=all_classes, yticklabels=all_classes,
                    cbar_kws={'label': 'Proportion prédite (Recall)'},
                    annot_kws={"size": 8})
        
        plt.title(f'Matrice de Confusion (Cumulée sur 25 Folds) - {method_name}', fontsize=16, fontweight='bold', pad=20)
        plt.ylabel('Vraie Espèce', fontsize=12, fontweight='bold')
        plt.xlabel('Espèce Prédite', fontsize=12, fontweight='bold')
        plt.xticks(rotation=45, ha='right', fontsize=9, fontstyle='italic')
        plt.yticks(fontsize=9, fontstyle='italic')
        
        plt.tight_layout()
        out_path = os.path.join(OUTPUT_DIR, f"cm_{method_name}.png")
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f"✅ Matrice générée et sauvegardée : {out_path}")
        
        # Affichage direct dans le notebook
        plt.show()

# Lancement de la fonction
plot_confusion_matrices()